# N03 — k-NN vs Decision Tree

**Pendamping S07 — Kecerdasan Buatan & ML Dasar**

Pola kerja: **DO → OBSERVE → INTERPRET → EXPLAIN**. Jalankan sel secara berurutan, catat evidence, lalu tuliskan interpretasi Anda.

## k-NN vs Decision Tree pada Iris

In [ ]:
# Import library yang dibutuhkan
import pandas as pd  # untuk membuat tabel hasil
from sklearn.datasets import load_iris  # dataset Iris bawaan scikit-learn
from sklearn.model_selection import train_test_split  # membagi data latih dan data uji
from sklearn.pipeline import Pipeline  # menggabungkan scaler dan model dalam satu alur
from sklearn.preprocessing import StandardScaler  # menyamakan skala fitur
from sklearn.neighbors import KNeighborsClassifier  # model k-NN (berbasis jarak)
from sklearn.tree import DecisionTreeClassifier  # model ai laDecision Tree (berbasis split)
from sklearn.metrics import accuracy_score  # metrik akurasi (tidak dipakngsung, karena .score() sudah menghitung akurasi)

# Load dataset Iris: 150 data, 4 fitur, 3 kelas
data=load_iris()
# Split 70% data latih : 30% data uji
# random_state=42 supaya hasil split selalu sama, stratify supaya proporsi tiap kelas di train dan test sama
X_train,X_test,y_train,y_test=train_test_split(
    data.data,data.target,test_size=.3,random_state=42,stratify=data.target
)

rows=[]  # list kosong untuk menyimpan hasil tiap model

# k-NN: coba k = 1, 5, 15
for k in [1,5,15]:
    # Pipeline: scaler di-fit hanya pada data latih lalu dipakai ke k-NN (mencegah data leakage)
    model=Pipeline([('scaler',StandardScaler()),('knn',KNeighborsClassifier(n_neighbors=k))])
    model.fit(X_train,y_train)  # latih model dengan data latih
    # simpan akurasi train dan test ke rows
    rows.append({'model':f'kNN k={k}','train':model.score(X_train,y_train),'test':model.score(X_test,y_test)})

# Decision Tree: coba max_depth = 2, 4, None (None = tanpa batas kedalaman)
for d in [2,4,None]:
    model=DecisionTreeClassifier(max_depth=d,random_state=42)  # tree tidak memakai scaler
    model.fit(X_train,y_train)  # latih tree dengan data latih
    # simpan akurasi train dan test ke rows
    rows.append({'model':f'Tree depth={d}','train':model.score(X_train,y_train),'test':model.score(X_test,y_test)})

# Tampilkan semua hasil dalam satu tabel
pd.DataFrame(rows)

,model,train,test
0,kNN k=1,1.000000,0.933333
1,kNN k=5,0.980952,0.911111
2,kNN k=15,0.980952,0.933333
3,Tree depth=2,0.971429,0.888889
4,Tree depth=4,0.990476,0.888889
5,Tree depth=None,1.000000,0.933333


**Tabel perbandingan score, preprocessing, complexity, dan interpretability.**


| Aspek | k-NN | Decision Tree |
|---|---|---|
| Score (test) | 0.9111 sampai 0.9333 | 0.8889 sampai 0.9333 |
| Preprocessing | Perlu scaling karena berbasis jarak (di Iris tidak menaikkan akurasi) | Tidak wajib karena berbasis split per fitur |
| Complexity | Hampir tidak ada proses latih, menyimpan semua data latih, prediksi harus menghitung jarak ke semua data. Diatur lewat k | Dibangun dari split berurutan, prediksi cepat (dari akar ke daun). Diatur lewat max_depth (3 sampai 8 daun di percobaan ini) |
| Interpretability | Rendah, tidak ada aturan yang jelas, hanya melihat tetangga terdekat | Tinggi, berupa aturan if-else yang bisa dibaca dan digambar |

### Evidence Log

| Eksperimen | Perubahan | Metric / Output | Interpretasi |
|---|---|---|---|
| EXP-01 | Ubah k pada k-NN (1, 5, 15), scaling tetap | Test 0.9333 / 0.9111 / 0.9333. Gap 0.0667 / 0.0698 / 0.0476 | k kecil lebih menghafal (train 1.0 di k=1), k besar lebih halus. Beda test hanya 1 data |
| EXP-02 | Ubah max_depth pada tree (2, 4, None) | Train 0.9714 / 0.9905 / 1.0. Test 0.8889 / 0.8889 / 0.9333. Daun 3 / 7 / 8 | Makin dalam makin kompleks dan makin menghafal data latih. Test tidak turun jelas di Iris |
| EXP-03 | k-NN tanpa scaling vs dengan scaling | k=1: 0.9333 vs 0.9333. k=5: 0.9778 vs 0.9111. k=15: 0.9556 vs 0.9333 | Scaling tidak menaikkan akurasi di Iris karena fitur sudah satuan cm. Beda 0 sampai 3 data |

### Tugas
1. Model mana yang memiliki gap train–test besar?
2. Apa efek k kecil/besar?
3. Apa efek depth?
4. Mengapa k-NN memakai scaling sementara tree umumnya tidak wajib?

### Sel tambahan: baseline dan k-NN tanpa scaling
Sel ini dipakai sebagai bukti untuk Task 1 (baseline) dan pertanyaan nomor 4 (scaling). Split data sama dengan sel di atas.

In [ ]:
import numpy as np  # untuk menghitung jumlah data per kelas
from sklearn.dummy import DummyClassifier  # model baseline sederhana

# Cek proporsi kelas setelah split stratified (kelas 0, 1, 2)
print('Jumlah data per kelas (train):', np.bincount(y_train))
print('Jumlah data per kelas (test) :', np.bincount(y_test))

# Baseline: selalu menebak kelas yang paling sering muncul di data latih
baseline=DummyClassifier(strategy='most_frequent')
baseline.fit(X_train,y_train)
print('Baseline -> train:',round(baseline.score(X_train,y_train),4),'| test:',round(baseline.score(X_test,y_test),4))

# Bandingkan k-NN tanpa scaling dan dengan scaling (split yang sama dengan sel sebelumnya)
rows2=[]
for k in [1,5,15]:
    knn_raw=KNeighborsClassifier(n_neighbors=k)  # tanpa scaling (data mentah)
    knn_raw.fit(X_train,y_train)
    knn_scaled=Pipeline([('scaler',StandardScaler()),('knn',KNeighborsClassifier(n_neighbors=k))])  # dengan scaling
    knn_scaled.fit(X_train,y_train)
    # simpan akurasi test kedua versi
    rows2.append({'k':k,
                  'test tanpa scaling':round(knn_raw.score(X_test,y_test),4),
                  'test dengan scaling':round(knn_scaled.score(X_test,y_test),4)})

# Tampilkan hasil perbandingan
pd.DataFrame(rows2)

Jumlah data per kelas (train): [35 35 35]
Jumlah data per kelas (test) : [15 15 15]
Baseline -> train: 0.3333 | test: 0.3333


,k,test tanpa scaling,test dengan scaling
0,1,0.9333,0.9333
1,5,0.9778,0.9111
2,15,0.9556,0.9333


### Jawaban Tugas

Gap = akurasi train − akurasi test.

| Model | Train | Test | Gap |
|---|---|---|---|
| kNN k=1 | 1.0000 | 0.9333 | 0.0667 |
| kNN k=5 | 0.9810 | 0.9111 | 0.0698 |
| kNN k=15 | 0.9810 | 0.9333 | 0.0476 |
| Tree depth=2 | 0.9714 | 0.8889 | 0.0825 |
| Tree depth=4 | 0.9905 | 0.8889 | 0.1016 |
| Tree depth=None | 1.0000 | 0.9333 | 0.0667 |

**1. Model dengan gap train–test besar**
Gap terbesar ada di Tree depth=4 (0.1016), lalu Tree depth=2 (0.0825). Gap terkecil ada di kNN k=15 (0.0476). Data test hanya 45, jadi 1 data salah sama dengan sekitar 2,2%. Selisih kecil belum tentu berarti.

**2. Efek k kecil/besar**
Pada k=1, akurasi train 1.0 karena tetangga terdekat dari tiap data latih adalah dirinya sendiri. Model jadi seperti menghafal dan batas kelasnya mudah terpengaruh noise (rawan overfitting). Pada k=15, prediksi diambil dari lebih banyak tetangga sehingga batasnya lebih halus. Train turun jadi 0.9810 dan gap mengecil jadi 0.0476. Kalau k terlalu besar, model bisa underfitting. Akurasi test tidak naik teratur (0.9333, 0.9111, 0.9333) dan bedanya cuma 1 data, jadi tidak bisa dibilang ada k yang paling bagus.

**3. Efek depth**
Makin dalam, tree makin kompleks. Depth=2 punya 3 daun (train 0.9714), depth=4 punya 7 daun (train 0.9905), dan depth=None punya 8 daun dengan kedalaman asli 5 (train 1.0, menghafal data latih). Akurasi test-nya 0.8889, 0.8889, 0.9333. Depth=2 paling sederhana dan cenderung underfitting. Pola overfitting yang biasa (test turun saat depth naik) tidak terlihat jelas di sini, mungkin karena Iris kecil dan mudah. Tapi train accuracy 1.0 tetap tidak bisa dijadikan bukti model bagus.

**4. Scaling pada k-NN dan tree**
k-NN menghitung jarak antar data, jadi fitur yang nilainya lebih besar akan lebih mendominasi jarak. Scaling menyamakan skala supaya semua fitur ikut berperan seimbang. Tree membagi data per satu fitur dengan threshold, dan scaling tidak mengubah urutan nilai, jadi hasil split-nya pada dasarnya sama dan scaling tidak wajib.
Dari sel tambahan, k=1 hasilnya sama (0.9333). Pada k=5, tanpa scaling 0.9778 dan dengan scaling 0.9111. Pada k=15, tanpa scaling 0.9556 dan dengan scaling 0.9333. Jadi di Iris scaling tidak menaikkan akurasi, malah sedikit lebih rendah (selisih 1 sampai 3 data). Kemungkinan karena semua fitur Iris sudah dalam cm dan skalanya tidak terlalu jauh. Pada data dengan satuan yang beda jauh (misalnya gaji dan umur), scaling tetap penting.

## Refleksi akhir

1. Apa hasil utama yang Anda amati?
2. Mengapa hasil tersebut terjadi menurut konsep di Main Deck?
3. Apa limitation eksperimen ini?
4. Eksperimen apa yang layak dilakukan berikutnya?

### Jawaban Refleksi

**1. Hasil utama**
Semua model jauh di atas baseline (0.3333). Akurasi test ada di 0.8889 sampai 0.9333 dan akurasi train di 0.9714 sampai 1.0. Gap terbesar ada di Tree depth=4 (0.1016) dan terkecil di kNN k=15 (0.0476). Scaling pada k-NN tidak menaikkan akurasi di Iris. Selisih antar model kecil sekali, hanya 1 sampai 3 data dari 45.

**2. Alasan hasil tersebut**
Hasilnya bisa dijelaskan dengan konsep overfitting, underfitting, dan kompleksitas model. k kecil dan depth besar membuat model lebih kompleks sehingga menghafal data latih (kNN k=1 dan Tree depth=None sama-sama train 1.0). k besar dan depth kecil membuat model lebih sederhana. Selain itu, k-NN berbasis jarak sehingga terpengaruh skala fitur, sedangkan tree berbasis split per fitur sehingga tidak.

**3. Limitation**
- Hanya satu kali split (random_state=42) dengan 45 data test, jadi hasil bisa berubah di split lain.
- Iris kecil, bersih, dan mudah dipisahkan, jadi efek k, depth, dan scaling belum tentu sama di data lain.
- Hanya memakai akurasi, dan hanya 3 nilai k dan 3 nilai depth.

**4. Eksperimen berikutnya**
- Pakai cross-validation atau ulangi dengan beberapa random_state supaya hasil lebih stabil.
- Coba lebih banyak nilai k dan max_depth, dipilih dengan validation set (bukan test set).
- Coba dataset yang skala fiturnya beda jauh untuk melihat pengaruh scaling yang lebih jelas.
- Tambah metrik lain seperti confusion matrix.